# TourismGPT — Phi-3 Mini Fine-tuning
Run on Google Colab with T4 GPU. Execute cells top to bottom.

# Fine-Tuning Microsoft Phi-3 Mini for TourismGPT

## Overview

This notebook focuses on fine-tuning Microsoft's Phi-3 Mini language model for the TourismGPT project. Instead of training a Large Language Model from scratch, a pre-trained model is adapted to the tourism domain using Parameter-Efficient Fine-Tuning (PEFT) with Quantized Low-Rank Adaptation (QLoRA).

The objective is to enable the model to understand and generate accurate responses to tourism-related queries while keeping computational requirements low enough to run efficiently on a Google Colab T4 GPU.

This notebook covers:
- Environment setup
- Loading the pre-trained Phi-3 Mini model
- Preparing the tourism instruction dataset
- Configuring QLoRA
- Fine-tuning the model
- Saving the trained adapter for later use in the TourismGPT chatbot

## Environment Setup

The required libraries are installed to support efficient model training, dataset processing, and parameter-efficient fine-tuning.

Key libraries include:

- Transformers
- Datasets
- PEFT
- Unsloth
- BitsAndBytes
- TRL

These tools enable fine-tuning of large language models while minimizing GPU memory usage.

In [1]:
# CELL 1 — Install dependencies (restart runtime if prompted)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-tnckzd2w/unsloth_ccfc503b161144278159b5b73baa1f78
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-tnckzd2w/unsloth_ccfc503b161144278159b5b73baa1f78
  Resolved https://github.com/unslothai/unsloth.git to commit 0c1c9f71dbc5842b92bdb4b1be65302838603870
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.6 MB/s eta 0:00:00
   

In [3]:
# CELL 2 — Mount Google Drive and copy dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# CELL 3 — Imports and config
import json
import os
import torch
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# ── Paths ──────────────────────────────────────────────────────────────────
JSONL_PATH   = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/datasets/tourism_finetune.jsonl"
OUTPUT_DIR   = "/content/tourism_gpt_adapter"
DRIVE_SAVE   = "/content/drive/MyDrive/tourism_gpt_adapter"

# ── Model config ───────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Phi-3-mini-4k-instruct"
MAX_SEQ_LEN  = 2048
DTYPE        = None
LOAD_IN_4BIT = True

# ── LoRA config ────────────────────────────────────────────────────────────
LORA_RANK    = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0

# ── Training config ────────────────────────────────────────────────────────
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
WARMUP_STEPS = 5
MAX_STEPS    = 300
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WEIGHT_DECAY = 0.01
SAVE_STEPS   = 100
LOGGING_STEPS = 25
FP16         = not torch.cuda.is_bf16_supported()
BF16         = torch.cuda.is_bf16_supported()
SEED         = 42

print("Config ready ✓")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Config ready ✓


## Loading the Pre-trained Model

Microsoft Phi-3 Mini is selected because it offers a strong balance between language understanding, reasoning capability, and computational efficiency.

Its compact architecture makes it suitable for fine-tuning on limited hardware while maintaining competitive performance on instruction-following tasks.

In [5]:
# CELL 4 — Load Phi-3 Mini with Unsloth (4-bit QLoRA)
print("Loading Phi-3 Mini ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

print(model.print_trainable_parameters())

Loading Phi-3 Mini ...
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 29,884,416 || all params: 3,850,963,968 || trainable%: 0.7760
None


## Preparing the Training Dataset

The model is trained using a tourism-specific instruction-response dataset containing travel-related questions and corresponding answers.

This supervised dataset enables the model to learn tourism terminology, destination knowledge, itinerary planning, accommodation guidance, transportation information, and other travel-related concepts.

In [6]:
import os # Added for path operations

# CELL 5 — Load and format dataset
EOS = tokenizer.eos_token

def format_example(row):
    instruction = row["instruction"].strip()
    output      = row["output"].strip()
    return (
        f"<|user|>\n{instruction}<|end|>\n"
        f"<|assistant|>\n{output}<|end|>\n"
        f"{EOS}"
    )

def load_jsonl(path):
    records = []
    # Check if the file exists before attempting to open it
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        parent_dir = os.path.dirname(path)
        if os.path.exists(parent_dir):
            print(f"Files found in '{parent_dir}': {os.listdir(parent_dir)}")
        else:
            print(f"The parent directory '{parent_dir}' does not exist. Please check your Google Drive mount and path.")
        raise FileNotFoundError(f"File not found: {path}") # Re-raise to stop execution

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"Loaded {len(records)} examples")
    texts = [format_example(r) for r in records]
    return Dataset.from_dict({"text": texts})

dataset = load_jsonl(JSONL_PATH)

print("\n── Sample training text ──────────────────────────────")
print(dataset["text"][0][:400])
print("──────────────────────────────────────────────────────")

Loaded 2220 examples

── Sample training text ──────────────────────────────
<|user|>
where can I check if there are any news on the rebate?<|end|>
<|assistant|>
I've observed that you're eager to stay updated on any news about your rebate. To check the latest updates, you can visit our website's "My Account" section. Log in using your credentials, and navigate to the "Rebate Status" or "Refund Status" page. This page will provide you with real-time information on the prog
──────────────────────────────────────────────────────


## Training Configuration

Training hyperparameters determine how the model learns from the tourism dataset.

The selected configuration balances training speed, memory usage, and model accuracy while remaining within the computational limits of Google Colab.

In [7]:
# CELL 6x — Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    args               = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        dataset_num_proc            = 2,
        packing                     = False,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        max_steps                   = MAX_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = FP16,
        bf16                        = BF16,
        logging_steps               = LOGGING_STEPS,
        optim                       = "adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 2,
        report_to                   = "none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU: {gpu_stats.name}  |  {max_memory} GB total")
print("\nStarting training ...")

trainer_stats = trainer.train()

print("\n── Training complete ──")
print(f"  Runtime  : {trainer_stats.metrics['train_runtime']:.0f} s")
print(f"  Samples/s: {trainer_stats.metrics['train_samples_per_second']:.1f}")
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"  Peak VRAM: {used_memory} GB / {max_memory} GB")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2220 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU: Tesla T4  |  14.563 GB total

Starting training ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,220 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.723251
50,1.019841
75,0.762693
100,0.663330
125,0.684788
150,0.683837
175,0.555852
200,0.624545
225,0.634108
250,0.606180


Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/checkpoint-300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter/checkpoint-300.



── Training complete ──
  Runtime  : 1084 s
  Samples/s: 2.2
  Peak VRAM: 2.816 GB / 14.563 GB


## Training Outcome

The fine-tuned Phi-3 Mini model has successfully learned tourism-specific knowledge from the instruction dataset.

The trained adapter weights are saved for use during inference and are later integrated with the Retrieval-Augmented Generation (RAG) pipeline to improve factual accuracy and contextual relevance.

In [8]:
# CELL 7 — Save LoRA adapter
print(f"Saving adapter to {OUTPUT_DIR} ...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved ✓")

# Copy to Google Drive so it survives after the session ends
import shutil
shutil.copytree(OUTPUT_DIR, DRIVE_SAVE, dirs_exist_ok=True)
print(f"Backed up to Drive: {DRIVE_SAVE} ✓")

Saving adapter to /content/tourism_gpt_adapter ...


Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter.


Adapter saved ✓
Backed up to Drive: /content/drive/MyDrive/tourism_gpt_adapter ✓


In [9]:
# CELL 8 — Quick inference test
def ask(question, max_new_tokens=300):
    FastLanguageModel.for_inference(model)
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_questions = [
    "Plan a 5-day itinerary for Tokyo for a solo budget traveller.",
    "Compare Bali vs Thailand for a honeymoon trip.",
    "What is the estimated budget for a week in Paris?",
    "How do I book a hotel with free cancellation on Booking.com?",
    "My flight was cancelled — how do I claim a refund?",
    "What cultural customs should I know before visiting Japan?",
]

print("── Inference tests ───────────────────────────────────")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)}")
    print("─" * 60)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


── Inference tests ───────────────────────────────────

Q: Plan a 5-day itinerary for Tokyo for a solo budget traveller.


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

A: Here is a 5-day itinerary for Tokyo tailored for a solo budget traveller:

Day 1–2: Arrive in Tokyo. Check into your hotel, recover from jet lag, and explore the city center. Visit the main landmarks and enjoy local street food.

Day 3–3: Head to Kyoto. Spend time exploring historical sites, markets, and local neighborhoods. Book a half-day guided tour for deeper cultural immersion.

Day 4–5: Travel to Osaka for the final stretch. Relax, enjoy nature or beach activities, and do any last-minute souvenir shopping.

Day 5: Depart. Allow at least 3 hours before your flight for airport transfer and check-in.

Tips for solo budget travellers: Book accommodations in advance, carry local currency, and always have a translation app handy.
────────────────────────────────────────────────────────────

Q: Compare Bali vs Thailand for a honeymoon trip.


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: For your query 'What is the estimated budget for a week in Paris?': Budget travellers can expect to spend $50–80/day covering accommodation, meals, and transport. Mid-range budgets of $100–150/day allow more comfort. Always set aside 10–15% for unexpected expenses.
────────────────────────────────────────────────────────────

Q: How do I book a hotel with free cancellation on Booking.com?


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Here is a suggested itinerary based on your query: How do I book a hotel with free cancellation on Booking.com? Day 1: Arrive and explore the city center. Day 2: Visit major landmarks. Day 3: Day trip to nearby attractions. Adjust based on your pace and interests.
────────────────────────────────────────────────────────────

Q: My flight was cancelled — how do I claim a refund?


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: I'm sorry to hear that your flight was cancelled. To claim a refund, you can follow these steps:

1. Check your travel insurance: If your flight was cancelled by the airline, check if your travel insurance covers the costs of the cancelled flight.
2. Contact the airline: Reach out to the airline directly and provide them with your booking details. They should offer a full refund within 21 days.
3. Travel refund claim form: Fill out the airline's travel refund claim form and submit it online or via mail.
4. Credit card chargeback: If you paid with a credit card, you may be eligible for a chargeback, which will refund the amount back to your bank account.
5. Travel agent or tour operator: If you booked through a travel agent or tour operator, contact them and follow their refund process.

Remember to keep all your receipts and booking confirmation emails for reference.
────────────────────────────────────────────────────────────

Q: What cultural customs should I know before visiting 

# Conclusion

This notebook demonstrates the complete fine-tuning workflow for adapting a pre-trained Large Language Model to the tourism industry.

By combining Phi-3 Mini with QLoRA, the project achieves efficient domain adaptation using limited computational resources. The resulting model forms the foundation of TourismGPT and is subsequently integrated with the RAG pipeline and Gradio interface to deliver accurate, industry-specific conversational responses.